# Skyline Online Course: Hypothesis Tests

Apply the hypothesis testing to the Skyline Online Courses dataset

Author: Meron Welderufael
Date: 08/17/2026

In [7]:
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations



In [8]:
enrollment = pd.read_csv('C:/Users/Lenovo/northstar-coursework/module-01-foundations-of-analytics-and-statistics/lesson-1-2-types-of-data/skyline_enrollments.csv')

python_for_beg = enrollment[enrollment["course_name"] == "Python for Beginners"]["hours_studied"]
sql_basics = enrollment[enrollment["course_name"] == "SQL Basics"]["hours_studied"]


print(f"python for beginner: n={len(python_for_beg) }, mean = {python_for_beg.mean():.4f}, std = {python_for_beg.std():.4f}")
print(f"sql basics: n={len(sql_basics)}, mean = {sql_basics.mean():.4f}, std = {sql_basics.std():.4f}")


python for beginner: n=34, mean = 29.9006, std = 2.9251
sql basics: n=32, mean = 29.8959, std = 3.9723


In [3]:
mean_diff = python_for_beg.mean() - sql_basics.mean()

#standard error of the difference

n1, n2 = len(python_for_beg) , len(sql_basics)
s1, s2 = python_for_beg.std(), python_for_beg.std()

se_diff = np.sqrt(s1**2/n1 + s2**2/n2)

# 95% CI for the difference
moe = 1.96 * se_diff
ci_lower = mean_diff - moe
ci_upper = mean_diff + moe

#p-value
t_stat, p_value = stats.ttest_ind(python_for_beg, sql_basics)


print(f"Mean diff(python for beg - sql basics) = {mean_diff:.4f}")
print(f"95% CI for the difference: ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"p-value: {p_value:.4f}")

Mean diff(python for beg - sql basics) = 0.0047
95% CI for the difference: (-1.41, 1.42)
p-value: 0.9957


“On average, Python for Beginners takes 0.0047 more hours than SQL Basics. Since 0.0047 hours is about 17 seconds, the difference between the two means is very small.”

The p-value of 0.9957 is greater than 0.05, so we fail to reject the null hypothesis and find no statistically significant difference in average study hours between Python for Beginners and SQL Basics.
The 95% confidence interval (−1.41, 1.42) includes 0, supporting the same conclusion.
The mean difference is only 0.0047 hours (about 17 seconds), indicating that the difference is very small and not practically meaningful.

In [30]:

n_tests = 10
results = []

courses = enrollment["course_name"].dropna().unique()

for course_a, course_b in combinations(courses, 2):
    group_a = enrollment.loc[enrollment["course_name"] == course_a]["hours_studied"].dropna()
    group_b = enrollment.loc[enrollment["course_name"] == course_b]["hours_studied"].dropna()

    t_stat, p_value = stats.ttest_ind(group_a, group_b)
    mean_diff = group_a.mean() - group_b.mean()

    results.append({
   "course_a": course_a,
   "course_b": course_b,
   "mean_difference": mean_diff,
   "p_value": p_value
    })

pairwise_results = pd.DataFrame(results)
pairwise_results


bonferroni_threshold = 0.05 / n_tests
significant_after_bonferroni = sum(
    result["p_value"] < bonferroni_threshold 
    for result in results)

print(f"Original threshold: 0.05")
print(f"Bonferroni-corrected threshold: {bonferroni_threshold:.4f}")
print(f"Number significant after Bonferroni: {significant_after_bonferroni}")




Original threshold: 0.05
Bonferroni-corrected threshold: 0.0050
Number significant after Bonferroni: 0


How many of the 10 tests came back with p < 0.05 (uncorrected)?

- 0 tests. All p-values are above 0.05, so none of the course pairs showed a statistically significant difference in average hours studied.

What is the Bonferroni-corrected threshold for 10 tests at the 0.05 family-wise error rate?
- 0.05 / 10 = 0.005. Therefore, the Bonferroni-corrected significance threshold is 0.005.

How many tests survive the Bonferroni correction?
- 0 tests. All p-values are above 0.005, so none are statistically significant after the correction.

Given that you generated this data with all courses drawn from similar distributions (the differences are random sampling noise), what does this tell you about the multiple testing trap?
- Because all courses were generated from similar distributions, the differences are mainly random sampling noise. Testing many pairs increases the chance of finding a seemingly significant result just by chance; the Bonferroni correction helps control this false-positive risk.

